<a href="https://colab.research.google.com/github/alfagalileo/Fisica-AC/blob/main/_Clase6.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab">
</a>

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

In [6]:
μ1, μ2 = 0.5, 0.5
m, M = 0.5, 5.0
g = 9.8

F_min = (m + M) * g * (1/μ2 + μ1)

def simular(F, T=10.0, npts=2000, y0=1.0):
    ax_est = (F - μ1*(M + m)*g) / (M + m)
    adhiere = μ2 * m * ax_est >= m * g

    if adhiere:
        ax, ay = ax_est, 0.0
    else:
        ax = (F - μ1*M*g)/( M + m + μ1*μ2*m )
        ay = μ2*ax - g

    ax = max(ax, 0.0)

    t = np.linspace(0.0, T, npts)
    dt = t[1] - t[0]

    r1, v1 = np.array([0.0, 0.0]), np.array([0.0, 0.0])
    r2, v2 = np.array([0.0, y0]), np.array([0.0, 0.0])
    R1, R2 = np.zeros((npts, 2)), np.zeros((npts, 2))

    nmax = 0
    for i in range(npts):
        R1[i], R2[i] = r1, r2
        nmax = i + 1

        if r2[1] < 0.0:
            break

        v1 = v1 + np.array([ax, 0.0])*dt
        r1 = r1 + v1*dt
        v2 = v2 + np.array([ax, ay])*dt
        r2 = r2 + v2*dt

    return t[:nmax], R1[:nmax], R2[:nmax], ax, ay, adhiere

def graficar(F):
    t, R1, R2, ax, ay, adhiere = simular(F)

    fig, (p1, p2) = plt.subplots(1, 2, figsize=(11, 4))

    p1.plot(t, R2[:, 1], lw=2, color="tab:red")
    p1.axhline(1.0, ls=":", c="gray")
    p1.axhline(0.0, ls="-", c="k", lw=1)
    p1.set_xlim(0, 10); p1.set_ylim(-0.05, 1.2)
    p1.set_xlabel("t [s]"); p1.set_ylabel("y de m [m]")
    p1.set_title("adherido" if adhiere else f"cae en t = {t[-1]:.2f} s")

    p2.plot(t, R1[:, 0], lw=2, label="M")
    p2.plot(t, R2[:, 0], lw=2, ls="--", label="m")
    p2.set_xlim(0, 10)
    p2.set_xlabel("t [s]"); p2.set_ylabel("x [m]")
    p2.legend()

    fig.suptitle(f"F = {F:.1f} N   (F_min = {F_min:.1f} N)   "
                 f"aₓ = {ax:.2f} m/s²   umbral g/μ₂ = {g/μ2:.2f} m/s²")
    plt.tight_layout()
    plt.show()

slider = widgets.FloatSlider(
    value=F_min + 1, min=0.0, max=2*F_min, step=1,
    description="F [N]:", continuous_update=False,
    readout_format=".1f", layout=widgets.Layout(width="600px")
)

display(widgets.interactive(graficar, F=slider))

interactive(children=(FloatSlider(value=135.75, continuous_update=False, description='F [N]:', layout=Layout(w…